In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv("../.env", override=True)

db_host = os.getenv("DB_HOST", "localhost")
db_port = os.getenv("DB_PORT", "5432")
db_name = os.getenv("DB_NAME", "postgres")
db_user = os.getenv("DB_USER", "postgres")
db_password = os.getenv("DB_PASSWORD", "postgres")

connection_string = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
conn = create_engine(connection_string)

## Projects that used test generalization

In [ ]:
import pandas as pd

query = f"SELECT id, root_path, runtime FROM project AS p WHERE p.use_test_generalization = true"
df = pd.read_sql_query(query, conn)
df

## Runtime requirements for test generalization per project

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re

# Define stage groups in the specified order
stage_groups = {
    'Original Validation': [
        'EXECUTE_TESTS_ORIGINAL', 'COLLECT_JUNIT_REPORTS_ORIGINAL',
        'COLLECT_JACOCO_DATA_ORIGINAL', 'FILTER_TESTS_ORIGINAL',
        'COLLECT_PIT_DATA_ORIGINAL'
    ],
    'Specification Extraction': [
        'BUILD_SPOON_MODEL', 'ANALYZE_TESTS', 'FILTER_TESTS', 'FILTER_ASSERTIONS',
        'ADD_JPF_INSTRUMENTATION', 'BUILD_PROJECT_INSTRUMENTED', 'EXECUTE_JPF', 'ANALYZE_JPF'
    ],
    'Initial Validation': [
        'ADD_DEPENDENCIES', 'BUILD_PROJECT_INITIAL', 'EXECUTE_TESTS_INITIAL',
        'COLLECT_JUNIT_REPORTS_INITIAL', 'COLLECT_JACOCO_DATA_INITIAL', 'COLLECT_PIT_DATA_INITIAL'
    ],
    'Generalization': [
        'GENERALIZE_TESTS'
    ],
    'Generalization Validation': [
        'BUILD_PROJECT_GENERALIZED', 'EXECUTE_TESTS_GENERALIZED',
        'COLLECT_JUNIT_REPORTS_GENERALIZED', 'FILTER_GENERALIZATIONS',
        'COLLECT_JACOCO_DATA_GENERALIZED', 'COLLECT_PIT_DATA_GENERALIZED'
    ],
    'Excluded': [
        'DOWNLOAD_PROJECT', 'SETUP_PROJECT', 'BUILD_PROJECT_ORIGINAL',
        'GENERATE_EVOSUITE_TESTS', 'POSTPROCESS_EVOSUITE_TESTS',
        'CLEANUP_PROJECT', 'CLEANUP_JPF_INSTRUMENTATION', 'CLEANUP_GENERALIZATION'
    ]
}

# Create a mapping from stage to group
stage_to_group = {stage: group for group, stages in stage_groups.items() for stage in stages}

# Query to get all tasks with runtime information
query = """
SELECT 
    p.id AS project_id,
    p.root_path,
    t.variant,
    t.stage,
    t.runtime
FROM 
    task t
JOIN 
    project p ON t.project_id = p.id
WHERE 
    t.runtime IS NOT NULL
"""

# Execute query and load into DataFrame
df_tasks = pd.read_sql_query(query, conn)

# Add stage group column and extract project name
df_tasks['stage_group'] = df_tasks['stage'].map(stage_to_group)
df_tasks['project_name'] = df_tasks['root_path'].apply(lambda path: path.split('/')[-1])

# Filter out excluded stages
df_tasks = df_tasks[df_tasks['stage_group'] != 'Excluded']

# Replace NULL variants with 'SHARED'
df_tasks['variant'] = df_tasks['variant'].fillna('SHARED')

# Get ordered groups (excluding 'Excluded')
ordered_groups = [g for g in stage_groups.keys() if g != 'Excluded']

# Extract base project names using regex to find the part before "-es-"
def get_base_project_name(project_name):
    # Match everything before "-es-" followed by any characters
    match = re.match(r'^(.*?)(?=-es-)', project_name)
    if match:
        return match.group(1)
    else:
        # Fallback: return the original name if pattern doesn't match
        return project_name

# Add base project name to the data
df_tasks['base_project_name'] = df_tasks['project_name'].apply(get_base_project_name)

# Aggregate data by project ID, project name, stage group, and variant
agg_data = df_tasks.groupby(['project_id', 'project_name', 'base_project_name', 'stage_group', 'variant'])['runtime'].sum().reset_index()

# Calculate total runtime per base project for sorting
base_project_totals = agg_data.groupby('base_project_name')['runtime'].sum().sort_values(ascending=False)
top_base_projects = base_project_totals.head(10).index.tolist()

# Filter for top 10 base projects
top_projects_data = agg_data[agg_data['base_project_name'].isin(top_base_projects)]

# Create a categorical type for stage_group to preserve order
top_projects_data['stage_group'] = pd.Categorical(
    top_projects_data['stage_group'], 
    categories=ordered_groups, 
    ordered=True
)

# Get all unique variants across all data
all_variants = sorted(df_tasks['variant'].unique())
non_shared_variants = [v for v in all_variants if v != 'SHARED']

# Create a color map with highly distinguishable colors
distinct_colors = [
    '#1f77b4',  # blue
    '#ff7f0e',  # orange
    '#2ca02c',  # green
    '#d62728',  # red
    '#9467bd',  # purple
    '#8c564b',  # brown
    '#e377c2',  # pink
    '#7f7f7f',  # gray
    '#bcbd22',  # olive
    '#17becf',  # cyan
    '#aec7e8',  # light blue
    '#ffbb78',  # light orange
    '#98df8a',  # light green
    '#ff9896',  # light red
    '#c5b0d5',  # light purple
]

# Ensure we have enough colors
if len(all_variants) > len(distinct_colors):
    additional_colors = plt.cm.Set3(np.linspace(0, 1, len(all_variants) - len(distinct_colors)))
    additional_colors = [tuple(c) for c in additional_colors]
    distinct_colors.extend(additional_colors)

# Create a mapping from variant to color
color_map = {variant: distinct_colors[i] for i, variant in enumerate(all_variants)}

# Create legend handles
legend_handles = [plt.Rectangle((0, 0), 1, 1, color=color_map[variant]) for variant in all_variants]
legend_labels = all_variants

# Helper function to add a horizontal legend at the top of a figure
def add_top_legend(fig, top_adjust=0.91):
    # Create the legend
    legend = fig.legend(
        legend_handles, 
        legend_labels, 
        loc='upper center', 
        ncol=min(len(all_variants), 5),
        frameon=False,
        bbox_to_anchor=(0.5, 0.98)  # Position the legend closer to the top
    )

    # Adjust the subplot positions to reduce space at the top
    plt.subplots_adjust(top=top_adjust)

    return legend

# Define which variants apply to which stage groups
stage_group_variants = {
    'Original Validation': ['SHARED'],
    'Specification Extraction': ['SHARED'],
    'Initial Validation': ['SHARED'],
    'Generalization': non_shared_variants,
    'Generalization Validation': non_shared_variants
}

# Get unique project IDs for plotting
unique_project_ids = top_projects_data['project_id'].unique()

# Define parameters for bar positioning
bar_width = 0.3       # Width of each bar
bar_spacing = 0.05    # Space between bars within a group
group_spacing = 0.3   # Space between different groups

# Count the number of bars in each group
bars_per_group = {group: len(variants) for group, variants in stage_group_variants.items()}

# Calculate the total width of each group (including internal bar spacing)
group_widths = {
    group: (count * bar_width) + ((count - 1) * bar_spacing) if count > 0 else 0
    for group, count in bars_per_group.items()
}

# Calculate the center position of each group
group_centers = {}
current_position = 0
for group in ordered_groups:
    width = group_widths[group]
    group_centers[group] = current_position + width / 2
    current_position += width + group_spacing

# Calculate the position of each bar within its group
bar_positions = {}
for group in ordered_groups:
    variants = stage_group_variants[group]
    num_bars = len(variants)

    if num_bars == 0:
        continue

    group_center = group_centers[group]
    group_width = group_widths[group]

    # Calculate the leftmost bar position
    start_pos = group_center - group_width / 2

    # Calculate position for each bar
    for i, variant in enumerate(variants):
        bar_positions[(group, variant)] = start_pos + i * (bar_width + bar_spacing) + bar_width / 2

# Create pivot tables for all projects to find the maximum bar height
all_pivot_tables = {}
for project_id in unique_project_ids:
    project_data = top_projects_data[top_projects_data['project_id'] == project_id]
    pivot_data = pd.pivot_table(
        project_data,
        index='stage_group',
        columns='variant',
        values='runtime',
        aggfunc='sum',
        fill_value=0,
        observed=False
    )
    all_pivot_tables[project_id] = pivot_data

# Find the maximum bar height for each base project
base_project_max_values = {}
for base_name in top_projects_data['base_project_name'].unique():
    max_value = 0
    for project_id in unique_project_ids:
        project_data = top_projects_data[top_projects_data['project_id'] == project_id]
        if project_data.empty:
            continue

        if project_data['base_project_name'].iloc[0] == base_name:
            pivot_data = all_pivot_tables[project_id]
            # Find the maximum value in the pivot table (this is the highest bar)
            if not pivot_data.empty:
                project_max = pivot_data.max().max()
                max_value = max(max_value, project_max)

    # Add some padding to the maximum value
    base_project_max_values[base_name] = max_value * 1.15

# Helper function to format runtime values
def format_runtime(seconds):
    if seconds < 10:
        return f"{seconds:.1f}"
    elif seconds < 100:
        return f"{seconds:.0f}"
    elif seconds < 1000:
        return f"{seconds:.0f}"
    else:
        return f"{seconds/1000:.1f}k"

# Plot 1: Runtime by Stage Group for each Project
fig1 = plt.figure(figsize=(16, 3*len(unique_project_ids)))

# Increase vertical spacing between subplots
plt.subplots_adjust(hspace=0.3)

# Plot each project ID
for i, project_id in enumerate(unique_project_ids):
    project_data = top_projects_data[top_projects_data['project_id'] == project_id]

    if project_data.empty:
        continue

    project_name = project_data['project_name'].iloc[0]
    base_name = project_data['base_project_name'].iloc[0]

    # Create a subplot for this project
    ax = plt.subplot(len(unique_project_ids), 1, i+1)

    # Get the pivot table for this project
    pivot_data = all_pivot_tables[project_id]

    # Plot each bar
    for group in ordered_groups:
        variants = stage_group_variants[group]
        for variant in variants:
            if variant in pivot_data.columns and group in pivot_data.index:
                value = pivot_data.loc[group, variant]
                position = bar_positions[(group, variant)]

                # Only plot and label bars with non-zero values
                if value > 0:
                    bar = ax.bar(
                        position,
                        value,
                        width=bar_width,
                        color=color_map[variant]
                    )

                    # Add value label on top of the bar
                    formatted_value = format_runtime(value)
                    ax.text(
                        position,                 # x position (center of bar)
                        value + (base_project_max_values[base_name] * 0.02),  # y position (slightly above bar)
                        formatted_value,          # text (formatted value)
                        ha='center',              # horizontal alignment
                        va='bottom',              # vertical alignment
                        fontsize=8,               # smaller font size
                        rotation=0                # no rotation
                    )

    ax.set_title(f'Project ID: {project_id} - {project_name}')
    ax.set_ylabel('Runtime (seconds)')

    # Set x-ticks at the center of each group
    ax.set_xticks([group_centers[group] for group in ordered_groups])

    # Only show x-tick labels for the last subplot
    if i == len(unique_project_ids) - 1:
        ax.set_xticklabels(ordered_groups, rotation=45, ha='right')
    else:
        ax.set_xticklabels([])  # Empty labels for all but the last subplot

    # Set x-limits to ensure proper padding
    ax.set_xlim(-0.5, current_position - group_spacing + 0.5)

    # Set y-limit based on the maximum value for this base project
    ax.set_ylim(0, base_project_max_values[base_name])

    # Add a grid for better readability
    ax.yaxis.grid(True, linestyle='--', alpha=0.7)

    # Remove individual legends
    if ax.get_legend() is not None:
        ax.get_legend().remove()

# Add the legend to the right side of the figure
legend1 = add_right_legend(fig1)

plt.tight_layout()  # Adjust layout to make sure everything fits
plt.subplots_adjust(right=0.85)  # Make sure this adjustment happens after tight_layout

plt.show()

In [ ]:
# Create a comprehensive DataFrame with all the data (excluding SHARED as separate rows)
def create_comprehensive_dataframe(df_tasks, top_base_projects):
    # Extract all project IDs from top base projects
    project_ids_in_top_base = df_tasks[df_tasks['base_project_name'].isin(top_base_projects)]['project_id'].unique()

    # Filter for projects in top base projects
    filtered_tasks = df_tasks[df_tasks['project_id'].isin(project_ids_in_top_base)]

    # Get all unique variants except SHARED
    non_shared_variants = sorted([v for v in filtered_tasks['variant'].unique() if v != 'SHARED'])

    # Create a list to hold rows for the new DataFrame
    rows = []

    # For each project ID
    for project_id in project_ids_in_top_base:
        project_data = filtered_tasks[filtered_tasks['project_id'] == project_id]

        # Skip if no data for this project ID
        if project_data.empty:
            continue

        project_name = project_data['project_name'].iloc[0]
        base_project_name = project_data['base_project_name'].iloc[0]

        # Get shared data (stages that are common to all variants)
        shared_data = project_data[project_data['variant'] == 'SHARED']

        # For each non-shared variant
        for variant in non_shared_variants:
            # Get variant-specific data
            variant_data = project_data[project_data['variant'] == variant]

            # Skip if this project doesn't have this variant
            if variant_data.empty:
                continue

            # Create a row dictionary
            row = {
                'project_id': project_id,
                'project_name': project_name,
                'base_project_name': base_project_name,
                'variant': variant
            }

            # Add runtime for each stage group
            for group in ordered_groups:
                # For shared stages, use data from SHARED variant
                if group in ['Original Validation', 'Specification Extraction', 'Initial Validation']:
                    group_data = shared_data[shared_data['stage_group'] == group]
                    row[group] = group_data['runtime'].sum() if not group_data.empty else 0
                # For variant-specific stages, use data from this variant
                else:
                    group_data = variant_data[variant_data['stage_group'] == group]
                    row[group] = group_data['runtime'].sum() if not group_data.empty else 0

            # Calculate total runtime
            row['Total Runtime'] = sum(row[group] for group in ordered_groups)

            rows.append(row)

    # Create DataFrame from rows
    result_df = pd.DataFrame(rows)

    # Set column order
    column_order = ['project_id', 'project_name', 'base_project_name', 'variant'] + ordered_groups + ['Total Runtime']
    result_df = result_df[column_order]

    # Sort by base project name, project name, and variant
    result_df = result_df.sort_values(['base_project_name', 'project_name', 'project_id', 'variant'])

    return result_df

# Create the comprehensive DataFrame
comprehensive_df = create_comprehensive_dataframe(df_tasks, top_base_projects)

# Display the DataFrame
display(comprehensive_df)
